# Benchmark — 6 datasets

Ce notebook execute le pipeline de preprocessing sur 6 datasets publics aux caracteristiques
volontairement contrastees. Chaque run est **auto-approuve** (pas d'interruption humaine).
Un tableau de synthese final compare les resultats.

| # | Dataset | Task | Lignes | Defis principaux |
|---|---------|------|--------|------------------|
| 1 | Titanic | classification | 891 | NaN, colonnes triviales, mix type |
| 2 | Diamonds | regression | 53940 | categoriques ordinales, volume |
| 3 | Penguins | classification | 344 | multi-classe, NaN, biologique |
| 4 | MPG | regression | 398 | colonne texte ID, NaN (horsepower) |
| 5 | Tips | regression | 244 | petit dataset, categoriques binaires |
| 6 | Healthexp | regression | 540 | serie temporelle, split temporel |

## 1. Setup

In [1]:
import json
import sqlite3
import time
import uuid
from pathlib import Path

import pandas as pd
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command

from src.agents.pipeline import build_pipeline

c:\Users\abdel\GenAI project\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
def _make_pipeline():
    """Instancie un pipeline avec SqliteSaver (checkpoint persistant sur disque)."""
    db_path = Path("data/cache/checkpoints.db")
    db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(str(db_path), check_same_thread=False)
    return build_pipeline(checkpointer=SqliteSaver(conn))


def run_auto(df_path: str, target: str, split_config: dict | None = None) -> dict:
    """Execute le pipeline complet avec auto-approve a chaque interruption.

    Retourne un dict avec : success, duration, domain_context, transform_proposals,
    leakage_alerts, outlier_proposals, quality_score, report.
    """
    pipeline = _make_pipeline()
    thread_id = str(uuid.uuid4())          # thread unique par run
    config = {"configurable": {"thread_id": thread_id}}
    out = {"df_path": df_path, "target": target, "errors": []}
    t0 = time.time()

    try:
        initial = {"df_path": df_path, "target": target, "split_config": split_config or {}}
        pipeline.invoke(initial, config=config)

        # --- Interrupt 1 : domain review ---
        snap = pipeline.get_state(config)
        iv = snap.tasks[0].interrupts[0].value
        out["domain_context"] = iv.get("domain_context", {})
        out["semantic_anomalies"] = iv.get("semantic_anomalies", [])
        pipeline.invoke(Command(resume="approve"), config=config)

        # --- Interrupt 2 : transformation review ---
        snap = pipeline.get_state(config)
        iv = snap.tasks[0].interrupts[0].value
        out["transform_proposals"] = iv.get("proposals", {})
        out["leakage_alerts"] = iv.get("leakage_alerts", [])
        out["correlation_alerts"] = iv.get("correlation_alerts", [])
        pipeline.invoke(Command(resume="approve"), config=config)

        # --- Interrupt 2b : leakage sub-interrupt (si present) ---
        snap = pipeline.get_state(config)
        if snap.tasks and snap.tasks[0].interrupts:
            iv = snap.tasks[0].interrupts[0].value
            if iv.get("type") == "leakage_warning":
                pipeline.invoke(Command(resume="approve"), config=config)
                snap = pipeline.get_state(config)

        # --- Interrupt 3 : outlier review ---
        if snap.tasks and snap.tasks[0].interrupts:
            iv = snap.tasks[0].interrupts[0].value
            out["outlier_proposals"] = iv.get("proposals", {})
            state = pipeline.invoke(Command(resume="approve"), config=config)
        else:
            state = snap.values

        out["report"] = state.get("report", {})
        out["quality_score"] = state.get("quality_score", {})
        out["confidence_map"] = state.get("confidence_map", [])
        out["success"] = True

    except Exception as e:
        out["success"] = False
        out["errors"].append(f"{type(e).__name__}: {e}")

    out["duration"] = round(time.time() - t0, 1)
    return out

## 2. Definition des 6 datasets

In [3]:
BASE = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master"

DATASETS = {
    "titanic": {
        "url": f"{BASE}/titanic.csv",
        "target": "survived",
        "split_config": {"strategy": "stratified"},
        "focus": "NaN, colonnes triviales (deck), mix type",
    },
    "diamonds": {
        "url": f"{BASE}/diamonds.csv",
        "target": "price",
        "split_config": {"strategy": "auto"},
        "focus": "categoriques ordinales (cut/color/clarity), 54k lignes",
    },
    "penguins": {
        "url": f"{BASE}/penguins.csv",
        "target": "species",
        "split_config": {"strategy": "stratified"},
        "focus": "multi-classe, NaN, colonnes biologiques",
    },
    "mpg": {
        "url": f"{BASE}/mpg.csv",
        "target": "mpg",
        "split_config": {"strategy": "auto"},
        "focus": "colonne texte ID (name), NaN (horsepower), origine categorique",
    },
    "tips": {
        "url": f"{BASE}/tips.csv",
        "target": "tip",
        "split_config": {"strategy": "auto"},
        "focus": "petit dataset (244 lignes), toutes colonnes cat sauf total_bill",
    },
    "healthexp": {
        "url": f"{BASE}/healthexp.csv",
        "target": "Spending_USD",
        "split_config": {"strategy": "temporal", "time_col": "Year"},
        "focus": "serie temporelle, split temporel sur Year",
    },
}

# Apercu de chaque dataset avant le pipeline
for name, cfg in DATASETS.items():
    try:
        df = pd.read_csv(cfg["url"])
        missing_pct = round(df.isnull().mean().mean() * 100, 1)
        print(f"{name:12} {df.shape[0]:>6} x {df.shape[1]:<3}  "
              f"target={cfg['target']:<18} NaN={missing_pct}%  focus: {cfg['focus']}")
    except Exception as e:
        print(f"{name:12} ERREUR: {e}")

titanic         891 x 15   target=survived           NaN=6.5%  focus: NaN, colonnes triviales (deck), mix type
diamonds      53940 x 10   target=price              NaN=0.0%  focus: categoriques ordinales (cut/color/clarity), 54k lignes
penguins        344 x 7    target=species            NaN=0.8%  focus: multi-classe, NaN, colonnes biologiques
mpg             398 x 9    target=mpg                NaN=0.2%  focus: colonne texte ID (name), NaN (horsepower), origine categorique
tips            244 x 7    target=tip                NaN=0.0%  focus: petit dataset (244 lignes), toutes colonnes cat sauf total_bill
healthexp       274 x 4    target=Spending_USD       NaN=0.0%  focus: serie temporelle, split temporel sur Year


## 3. Execution du benchmark

Chaque dataset est execute sequentiellement avec auto-approve. Les resultats sont stockes dans `RESULTS`.

> Le pipeline fait plusieurs appels LLM par dataset (domaine, transformations, outliers, rapport).
> Duree estimee : **5 a 15 min** selon les temps de reponse OpenAI.

In [4]:
RESULTS = {}

for name, cfg in DATASETS.items():
    print(f"[{name}] lancement...")
    r = run_auto(cfg["url"], cfg["target"], split_config=cfg.get("split_config"))
    RESULTS[name] = r
    status = "OK" if r["success"] else f"ERREUR: {r['errors']}"
    qs = r.get("quality_score", {})
    overall = qs.get("overall", "?") if r["success"] else "-"
    print(f"[{name}] {status} | score={overall}/100 | duree={r['duration']}s")
    print()

[titanic] lancement...


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: latifo (latifo-universit-paris-dauphine-psl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Weave is installed but not imported. Add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
20:05:55 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=titanic
20:05:56 - src.agents.base_agent - INFO - Dataset loaded: 891 rows x 15 columns, target='survived'
20:05:56 - src.agents.base_agent - INFO - Target 'survived': task_type=classification, 2 unique, 0.0% NaN
c:\Users\abdel\GenAI project\src\agents\base_agent.py:102: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_count = int(pd.to_datetime(non_null, errors="coerce").notna().sum())
c:\Users\abdel\GenAI project\src\agents\base_agent.py:102: UserWarning: Could not infer format, so each element will be

baseline/baseline_score,▁
baseline/delta,▁
baseline/model_score,▁
report/completeness,▁
report/deterministic_ratio,▁
report/outlier_coverage,▁
report/quality_overall,▁
report/type_consistency,▁
baseline/baseline_score,0.6145
baseline/delta,0.3855
baseline/model_score,1


20:07:22 - src.tracking.wandb_tracker - INFO - W&B run finished


...

[tips] ERREUR: ['ValueError: Could not extract valid JSON from LLM response:\n{\n  "dataset_summary": {\n    "original_shape": {\n      "rows": 244,\n      "cols": 7\n    },\n    "final_shape_reported": {\n      "rows": 195,\n      "cols": 9\n    },\n    "columns": {\n      "total_bill": {\n        "type": "numeric_continuous",\n        "min": 3.07,\n        "max": 50.81,\n        "skewness": 1.133\n      },\n      "tip": {\n        "type": "numeric_continuous (target)",\n        "nunique": 123,\n        "pct_nan": 0.0,\n        "mean": 2.9983,\n        "std": 1.3836\n      },\n      "sex": {\n  '] | score=-/100 | duree=71.8s

[healthexp] lancement...


20:09:06 - src.tracking.wandb_tracker - INFO - W&B initialized: project=preprocessing-pipeline, run=healthexp
20:09:06 - src.agents.base_agent - INFO - Dataset loaded: 274 rows x 4 columns, target='Spending_USD'
20:09:06 - src.agents.base_agent - INFO - Target 'Spending_USD': task_type=regression, 274 unique, 0.0% NaN
c:\Users\abdel\GenAI project\src\agents\base_agent.py:102: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_count = int(pd.to_datetime(non_null, errors="coerce").notna().sum())
20:09:06 - src.agents.base_agent - INFO - Inferring domain context and anomalies (1 LLM call)
20:09:06 - src.agents.base_agent - INFO - Generating diagnosis (1 LLM call)
20:09:06 - src.handlers.error_handler - WARNING - Validation failed (attempt 1/3): - Champ 'issues -> 0': Input should be a valid string (valeur recue: {'issue': 'panel_déséquilibré / observa

baseline/baseline_score,▁
baseline/delta,▁
baseline/model_score,▁
report/completeness,▁
report/deterministic_ratio,▁
report/outlier_coverage,▁
report/quality_overall,▁
report/type_consistency,▁
baseline/baseline_score,-3.1557
baseline/delta,3.8367
baseline/model_score,0.681


20:09:12 - src.tracking.wandb_tracker - INFO - W&B run finished


[healthexp] OK | score=100.0/100 | duree=9.3s



## 4. Tableau de synthese

In [5]:
rows = []
for name, cfg in DATASETS.items():
    r = RESULTS.get(name, {})
    qs = r.get("quality_score", {})
    domain = r.get("domain_context", {})
    transforms = r.get("transform_proposals", {}).get("transformations", [])
    n_llm = sum(1 for t in transforms if t.get("source") == "llm")
    n_rule = sum(1 for t in transforms if t.get("source") == "rule")
    leakage = len(r.get("leakage_alerts", []))
    corr_alerts = len(r.get("correlation_alerts", []))
    outlier_actions = r.get("outlier_proposals", {}).get("outlier_actions", [])
    n_clip = sum(1 for a in outlier_actions if a.get("action") == "clip")
    n_remove = sum(1 for a in outlier_actions if a.get("action") == "remove")
    n_keep = sum(1 for a in outlier_actions if a.get("action") == "keep")
    report = r.get("report", {})
    baseline = report.get("baseline_evaluation", {})

    rows.append({
        "Dataset": name,
        "Succes": "oui" if r.get("success") else "non",
        "Duree (s)": r.get("duration", "-"),
        "Domaine infere": domain.get("domain", "-"),
        "Score global": qs.get("overall", "-"),
        "Completude /30": qs.get("completeness", "-"),
        "Types /15": qs.get("type_consistency", "-"),
        "Outliers /20": qs.get("outlier_coverage", "-"),
        "Transforms rule": n_rule,
        "Transforms LLM": n_llm,
        "Leakage alerts": leakage,
        "Corr alerts": corr_alerts,
        "Outliers clip": n_clip,
        "Outliers remove": n_remove,
        "Outliers keep": n_keep,
        "Baseline metric": baseline.get("metric", "-"),
        "Baseline score": baseline.get("baseline_score", "-"),
        "Model score": baseline.get("model_score", "-"),
        "Delta": baseline.get("delta", "-"),
    })

summary_df = pd.DataFrame(rows).set_index("Dataset")
summary_df

,Succes,Duree (s),Domaine infere,Score global,Completude /30,Types /15,Outliers /20,Transforms rule,Transforms LLM,Leakage alerts,Corr alerts,Outliers clip,Outliers remove,Outliers keep,Baseline metric,Baseline score,Model score,Delta
Dataset,,,,,,,,,,,,,,,,,,
titanic,oui,102.0,transport,81.0,30.0,6.0,15.0,15,11,0,4,0,0,20,accuracy,0.6145,1.0,0.3855
diamonds,non,12.9,e-commerce,-,-,-,-,12,0,0,6,3,2,18,-,-,-,-
penguins,oui,7.3,autre,100.0,30.0,15.0,20.0,11,0,0,1,0,0,9,accuracy,0.4348,1.0,0.5652
mpg,oui,8.3,transport,97.0,30.0,12.0,20.0,9,0,0,7,1,0,8,r2,-0.004,0.8564,0.8604
tips,non,71.8,autre,-,-,-,-,6,0,0,1,0,0,8,-,-,-,-
healthexp,oui,9.3,sante,100.0,30.0,15.0,20.0,3,0,0,1,0,0,5,r2,-3.1557,0.681,3.8367


## 5. Analyse par dataset

### 5.1 Titanic

In [6]:
r = RESULTS["titanic"]
domain = r.get("domain_context", {})
qs = r.get("quality_score", {})

print(f"Domaine : {domain.get('domain')} - {domain.get('description', '')[:120]}")
print(f"Score : {qs.get('overall')}/100  "
      f"(completude={qs.get('completeness')}, types={qs.get('type_consistency')}, "
      f"outliers={qs.get('outlier_coverage')})")

anomalies = r.get("semantic_anomalies", [])
if anomalies:
    print(f"\nAnomalies semantiques ({len(anomalies)}) :")
    for a in anomalies:
        print(f"  {a['column']}: {a['issue']} [{a['severity']}]")

transforms = r.get("transform_proposals", {}).get("transformations", [])
print(f"\nTransformations proposees ({len(transforms)}) :")
for t in transforms:
    print(f"  [{t.get('source','?')}/{t.get('confidence','?')}] {t['column']}: {t['action']}")

Domaine : transport - Jeu de données de manifestes passagers (jeu bien connu du Titanic) décrivant caractéristiques démographiques et de voyag
Score : 81.0/100  (completude=30.0, types=6.0, outliers=15.0)

Anomalies semantiques (9) :
  age: Taux de valeurs manquantes important (~19.9%). Les ages manquants peuvent biaiser modèles et indicateurs démographiques. [high]
  deck: Très forte proportion de valeurs manquantes (~77.2%). beaucoup d'informations de cabine absentes. [high]
  embarked: Présence de valeurs manquantes (faible ~0.2) — incohérences possibles entre 'embarked' et 'embark_town'. [low]
  fare: Valeurs nulles (fare == 0.0) et fortes asymétries avec outliers (max ~512), susceptibles d'être tickets gratuits/erreurs ou outliers influents. [medium]
  alive: Colonne redondante avec la cible 'survived' (mappage textuel 'yes'/'no'). Risque de fuite de cible si utilisée telle quelle. [high]
  class: Colonne redondante avec 'pclass' (mêmes informations sous forme textuelle) — risque 

### 5.2 Diamonds

In [7]:
r = RESULTS["diamonds"]
domain = r.get("domain_context", {})
qs = r.get("quality_score", {})

print(f"Domaine : {domain.get('domain')} - {domain.get('description', '')[:120]}")
print(f"Score : {qs.get('overall')}/100")

corr = r.get("correlation_alerts", [])
if corr:
    print(f"\nCorrelations inter-features detectees ({len(corr)}) :")
    for c in corr[:5]:
        print(f"  {c['column_a']} <-> {c['column_b']}: {c['correlation']:.3f} [{c['severity']}]")

outliers = r.get("outlier_proposals", {}).get("outlier_report", [])
if outliers:
    print(f"\nRapport outliers ({len(outliers)} colonnes analysees) :")
    for o in outliers[:5]:
        print(f"  {o['column']}: {o['n_outliers']} outliers ({o['pct_outliers']}%) methode={o['method']}")

Domaine : e-commerce - Jeu de données sur des diamants (carat, qualité de taille/ couleur/clarity, dimensions x/y/z, depth/table) avec le prix 
Score : None/100

Correlations inter-features detectees (6) :
  carat <-> x: 0.975 [very_high]
  carat <-> y: 0.946 [very_high]
  carat <-> z: 0.948 [very_high]
  x <-> y: 0.969 [very_high]
  x <-> z: 0.966 [very_high]

Rapport outliers (23 colonnes analysees) :
  carat: 1510 outliers (3.5%) methode=iqr
  depth: 2020 outliers (4.7%) methode=iqr
  table: 479 outliers (1.1%) methode=iqr
  x: 18 outliers (0.0%) methode=iqr
  y: 18 outliers (0.0%) methode=iqr


### 5.3 Penguins

In [8]:
r = RESULTS["penguins"]
domain = r.get("domain_context", {})
qs = r.get("quality_score", {})

print(f"Domaine : {domain.get('domain')} - {domain.get('description', '')[:120]}")
print(f"Score : {qs.get('overall')}/100")
print(f"Colonnes sensibles : {domain.get('sensitive_columns', [])}")

report = r.get("report", {})
baseline = report.get("baseline_evaluation", {})
if baseline.get("status") == "ok":
    print(f"\nBaseline {baseline['metric']} :")
    print(f"  DummyClassifier : {baseline['baseline_score']}")
    print(f"  LogisticRegression : {baseline['model_score']}")
    print(f"  Delta : +{baseline['delta']}")

transforms = r.get("transform_proposals", {}).get("transformations", [])
print(f"\nTransformations ({len(transforms)}) :")
for t in transforms:
    print(f"  {t['column']}: {t['action']} ({t.get('source','?')})")

Domaine : autre - Données biologiques/ornithologiques (mesures morphométriques de manchots : espèces, île d'origine, longueurs du bec et d
Score : 100.0/100
Colonnes sensibles : []

Baseline accuracy :
  DummyClassifier : 0.4348
  LogisticRegression : 1.0
  Delta : +0.5652

Transformations (11) :
  bill_length_mm: impute_median (rule)
  bill_depth_mm: impute_median (rule)
  flipper_length_mm: impute_median (rule)
  body_mass_g: impute_median (rule)
  sex: impute_mode (rule)
  island: one_hot_encoding (rule)
  sex: ordinal_encoding (rule)
  bill_length_mm: standard_scaling (rule)
  bill_depth_mm: standard_scaling (rule)
  flipper_length_mm: standard_scaling (rule)
  body_mass_g: standard_scaling (rule)


### 5.4 MPG

In [9]:
r = RESULTS["mpg"]
domain = r.get("domain_context", {})
qs = r.get("quality_score", {})

print(f"Domaine : {domain.get('domain')} - {domain.get('description', '')[:120]}")
print(f"Score : {qs.get('overall')}/100")

transforms = r.get("transform_proposals", {}).get("transformations", [])
# Focus : comment le pipeline a traite la colonne 'name' (ID-like texte)
name_transform = [t for t in transforms if t.get("column") == "name"]
if name_transform:
    print(f"\nColonne 'name' (ID texte) traitee comme : {name_transform[0]['action']}")
    print(f"  Raison : {name_transform[0].get('reason', '?')}")
else:
    print("\nColonne 'name' : aucune transformation proposee (probablement droppee comme ID unique)")

# Colonne horsepower avec NaN
hp_transform = [t for t in transforms if t.get("column") == "horsepower"]
if hp_transform:
    for t in hp_transform:
        print(f"Colonne 'horsepower' : {t['action']} ({t.get('source','?')})")

Domaine : transport - Jeu de données automobile décrivant l'efficacité énergétique (mpg) de véhicules avec leurs caractéristiques techniques (
Score : 97.0/100

Colonne 'name' (ID texte) traitee comme : frequency_encoding
  Raison : high cardinality (246)
Colonne 'horsepower' : impute_median (rule)
Colonne 'horsepower' : robust_scaling (rule)


### 5.5 Tips

In [10]:
r = RESULTS["tips"]
domain = r.get("domain_context", {})
qs = r.get("quality_score", {})

print(f"Domaine : {domain.get('domain')} - {domain.get('description', '')[:120]}")
print(f"Score : {qs.get('overall')}/100")

report = r.get("report", {})
baseline = report.get("baseline_evaluation", {})
if baseline.get("status") == "ok":
    print(f"\nBaseline {baseline['metric']} :")
    print(f"  Dummy : {baseline['baseline_score']}  |  Ridge : {baseline['model_score']}  |  Delta : {baseline['delta']}")

# Contraintes metier detectees
constraints = domain.get("constraints", [])
if constraints:
    print(f"\nContraintes metier detectees :")
    for c in constraints:
        print(f"  - {c}")

cv = report.get("constraint_violations", [])
if cv:
    print(f"\nViolations de contraintes ({len(cv)}) :")
    for v in cv:
        print(f"  {v['constraint']}: {v['n_violations']} violations ({v['pct']}%)")
else:
    print("\nAucune violation de contrainte detectee.")

Domaine : autre - Données de facturation et de pourboire en restauration (montant de l'addition, pourboire, sexe du payeur, statut fumeur,
Score : None/100

Contraintes metier detectees :
  - total_bill doit être positif (> 0)
  - tip doit être >= 0 (pas de pourboire négatif)
  - généralement tip <= total_bill (vérifier les cas où tip >> total_bill)
  - size doit être un entier >= 1 (taille de table en nombre de personnes)
  - day doit appartenir à l'ensemble attendu (ex: Thu, Fri, Sat, Sun) et time à {Lunch, Dinner}
  - sex doit être dans {Male, Female} et smoker dans {Yes, No} (selon le codage utilisé)

Aucune violation de contrainte detectee.


### 5.6 Healthexp (split temporel)

In [11]:
r = RESULTS["healthexp"]
domain = r.get("domain_context", {})
qs = r.get("quality_score", {})

print(f"Domaine : {domain.get('domain')} - {domain.get('description', '')[:120]}")
print(f"Score : {qs.get('overall')}/100")

report = r.get("report", {})
split_info = report.get("train_test_split", {})
if split_info:
    print(f"\nSplit temporel :")
    print(f"  Train : {split_info.get('train_size', '?')} lignes")
    print(f"  Test  : {split_info.get('test_size', '?')} lignes")

transforms = r.get("transform_proposals", {}).get("transformations", [])
print(f"\nTransformations ({len(transforms)}) :")
for t in transforms:
    print(f"  {t['column']}: {t['action']} ({t.get('source','?')})")

Domaine : sante - Série temporelle par pays des dépenses en USD (cible) et de l'espérance de vie entre 1970 et 2020. Probablement dépenses
Score : 100.0/100

Split temporel :
  Train : 219 lignes
  Test  : 55 lignes

Transformations (3) :
  Country: one_hot_encoding (rule)
  Year: standard_scaling (rule)
  Life_Expectancy: standard_scaling (rule)


## 6. Comparaison des scores qualite

In [12]:
score_rows = []
for name, r in RESULTS.items():
    if not r.get("success"):
        continue
    qs = r.get("quality_score", {})
    report = r.get("report", {})
    baseline = report.get("baseline_evaluation", {})
    score_rows.append({
        "Dataset": name,
        "Overall /100": qs.get("overall"),
        "Completude /30": qs.get("completeness"),
        "Types /15": qs.get("type_consistency"),
        "Outliers /20": qs.get("outlier_coverage"),
        "Leakage": qs.get("leakage_risk"),
        "Baseline": baseline.get("baseline_score", "-"),
        "Model": baseline.get("model_score", "-"),
        "Delta": baseline.get("delta", "-"),
        "Version prompts": report.get("prompt_templates_version", "-"),
    })

scores_df = pd.DataFrame(score_rows).set_index("Dataset")
scores_df

,Overall /100,Completude /30,Types /15,Outliers /20,Leakage,Baseline,Model,Delta,Version prompts
Dataset,,,,,,,,,
titanic,81.0,30.0,6.0,15.0,low,0.6145,1.0000,0.3855,1.1
penguins,100.0,30.0,15.0,20.0,low,0.4348,1.0000,0.5652,1.1
mpg,97.0,30.0,12.0,20.0,low,-0.0040,0.8564,0.8604,1.1
healthexp,100.0,30.0,15.0,20.0,low,-3.1557,0.6810,3.8367,1.1


## 7. Sauvegarde des resultats

In [13]:
out_dir = Path("data/outputs/benchmark")
out_dir.mkdir(parents=True, exist_ok=True)

# Tableau de synthese
scores_df.to_csv(out_dir / "benchmark_scores.csv")
print(f"Scores sauvegardes : {out_dir / 'benchmark_scores.csv'}")

# Rapport complet de chaque dataset
for name, r in RESULTS.items():
    report = r.get("report")
    if report:
        path = out_dir / f"{name}_report.json"
        with open(path, "w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
        print(f"  {name}: rapport sauvegarde -> {path}")

Scores sauvegardes : data\outputs\benchmark\benchmark_scores.csv
  titanic: rapport sauvegarde -> data\outputs\benchmark\titanic_report.json
  penguins: rapport sauvegarde -> data\outputs\benchmark\penguins_report.json
  mpg: rapport sauvegarde -> data\outputs\benchmark\mpg_report.json
  healthexp: rapport sauvegarde -> data\outputs\benchmark\healthexp_report.json
